# Attribution recall

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
model_dir = './experiments/dnn_spikein'
attrs_fn = 'attributions_global_baselines_mode.csv'
data_dir = './data/spikein/spikein_s6_add100_dom100_rec100_epi100_seed42'
gwas_fn = 'gwas_real_plus_syn_train.height_adj_z.glm.linear'
attrs_path = os.path.join(model_dir, attrs_fn)
gwas_path = os.path.join(data_dir, gwas_fn)

In [ ]:
attrs_df = pd.read_csv(attrs_path, index_col='snp')
algo_names = list(attrs_df.columns)

print(f"Attributions file: {attrs_path}")
print(f"Attributed {len(attrs_df)} SNPs across {len(algo_names)} methods.")
print(algo_names)

In [ ]:
snp_col = 'ID' # snp names/identifiers column
p_col = 'P' # p-value column
alpha = 0.05 # sig level for bonferroni correction (alpha=0.05 for 95% confidence)
#bonf_thresh = 5e-8 # optional fixed significance threshold
gwas = pd.read_csv(gwas_path, sep='\t', dtype={'#CHROM':str})
bonf_thresh = alpha / gwas['P'].notna().sum()
gwas_sig_snps = set(gwas.loc[gwas[p_col] <= bonf_thresh, snp_col])

gwas_sig_syn_snps = set(gwas.loc[(gwas['#CHROM']=='MT') & (gwas[p_col] <= bonf_thresh), snp_col])
gwas_sig_syn_add = [s for s in gwas_sig_syn_snps if s.startswith('SYN_ADD_')]
gwas_sig_syn_dom = [s for s in gwas_sig_syn_snps if s.startswith('SYN_DOM_')]
gwas_sig_syn_rec = [s for s in gwas_sig_syn_snps if s.startswith('SYN_REC_')]
gwas_sig_syn_epi = [s for s in gwas_sig_syn_snps if s.startswith('SYN_EPI_')]

gwas['p_rank'] = gwas[p_col].rank().astype(int)
avg_add_snp_rank = gwas.loc[gwas[snp_col].str.startswith('SYN_ADD_'), 'p_rank'].mean()
avg_dom_snp_rank = gwas.loc[gwas[snp_col].str.startswith('SYN_DOM_'), 'p_rank'].mean()
avg_rec_snp_rank = gwas.loc[gwas[snp_col].str.startswith('SYN_REC_'), 'p_rank'].mean()
avg_epi_snp_rank = gwas.loc[gwas[snp_col].str.startswith('SYN_EPI_'), 'p_rank'].mean()

print(f"------------------- GWAS Report ------------------")
print(f"File: .../{'/'.join(gwas_path.split('/')[-2:])}")
print(f"Total number of SNPs: {gwas[snp_col].nunique()} "
      f"(n_real={len(set(gwas.loc[gwas['#CHROM']!='MT', snp_col]))}, "
      f"n_synthetic={len(set(gwas.loc[gwas['#CHROM']=='MT', snp_col]))})")
print(f"Bonferroni-corrected significance threshold (alpha={alpha:.3g}): {bonf_thresh:.3g}")
print(f"Total number of significant SNPs: {len(gwas_sig_snps)}")
print(f"Total number of significant synthetic SNPs: {len(gwas_sig_syn_snps)}")
print(f"- sig syn ADD snps: {len(gwas_sig_syn_add):>5d}, avg p-value rank: {avg_add_snp_rank.round():>8,.0f}")
print(f"- sig syn DOM snps: {len(gwas_sig_syn_dom):>5d}, avg p-value rank: {avg_dom_snp_rank.round():>8,.0f}")
print(f"- sig syn REC snps: {len(gwas_sig_syn_rec):>5d}, avg p-value rank: {avg_rec_snp_rank.round():>8,.0f}")
print(f"- sig syn EPI snps: {len(gwas_sig_syn_epi):>5d}, avg p-value rank: {avg_epi_snp_rank.round():>8,.0f}")
print('-'*50)
# Combine GWAS results with dnn attribution scores.
# gwas score is defined as -log p_value
gwas = pd.read_csv(gwas_path, sep='\t', dtype={'#CHROM':str})
gwas['gwas'] = -1 * np.log10(gwas[p_col])
scores_df = pd.merge(
    attrs_df, gwas[[snp_col, 'gwas']].set_index(snp_col), 
    left_index=True, right_index=True
)
gwas.drop(columns=['gwas'], inplace=True)

In [ ]:
def group_epistatic_snps(epi_snps):
    groups = {} # empty dict to hold groupings
    # loop through all possible epistatic SNPs
    for s in sorted(epi_snps):
        key = "_".join(s.split('_')[:3])
        if key not in groups:
            # new empty list for this group
            groups[key] = []
        # append SNP to this group
        groups[key].append(str(s))
    return groups

all_snps = set(scores_df.index.values.astype(str))
syn_add_snps = set([s for s in all_snps if s.startswith('SYN_ADD_')])
syn_dom_snps = set([s for s in all_snps if s.startswith('SYN_DOM_')])
syn_rec_snps = set([s for s in all_snps if s.startswith('SYN_REC_')])
syn_epi_snps = set([s for s in all_snps if s.startswith('SYN_EPI_')])

# epi_groups_dict is a dict where each key is the epi SNP pair base name (e.g., 'SYN_EPI_001') 
# and the value is a list of the two SNPs in that pair (e.g., ['SYN_EPI_001_A', 'SYN_EPI_001_B'])
epi_groups_dict = group_epistatic_snps(syn_epi_snps)
# epi_groups is a list of tuples, where each tuple contains the two SNPs 
# in an epistatic pair (sorted alphabetically)
epi_groups = [tuple(sorted(epi_groups_dict[key])) for key in sorted(epi_groups_dict.keys())]
n_total_epi_groups = len(epi_groups)

syn_snps = syn_add_snps | syn_dom_snps | syn_rec_snps | syn_epi_snps
syn_nonlinear_snps = syn_dom_snps | syn_rec_snps | syn_epi_snps
real_snps = all_snps - (syn_add_snps | syn_nonlinear_snps)

print(f"Total number of interpretation scores: {len(all_snps)}")
print(f"- Real SNPs: {len(real_snps)}")
print(f"- Synthetic additive SNPs: {len(syn_add_snps)}")
print(f"- Synthetic nonlinear SNPs: {len(syn_nonlinear_snps)}")
print(f"  - Dominant: {len(syn_dom_snps)}")
print(f"  - Recessive: {len(syn_rec_snps)}")
print(f"  - Epistatic: {len(syn_epi_snps)} (n_groupings={len(epi_groups)})")

In [ ]:
n_scores = len(scores_df)
q_start, q_end, q_step = 0.80, 1.0, 0.01
quantile_list = np.arange(q_start, q_end, q_step)

In [ ]:
results = []
for q in quantile_list:
    k = int(np.round((1-q)*n_scores))
    percentile = 100 * q
    print(f"quantile {q:.5f} |  percentile {percentile/100:.3%} | k={k:,}")
    for algo in scores_df.columns:
        score_by_snp = scores_df[algo]
        topk_snps = set(score_by_snp.nlargest(k).index)
        n_real = len(topk_snps & real_snps)
        n_syn = len(topk_snps & syn_snps) # synthetic (all synthetic snps are causal)
        n_gwas = len(topk_snps & gwas_sig_snps) # topk gwas significant snps
        n_syn_add = len(topk_snps & syn_add_snps) # topk synthetic additive
        n_syn_dom = len(topk_snps & syn_dom_snps) # topk synthetic dominant
        n_syn_rec = len(topk_snps & syn_rec_snps) # topk synthetic recessive
        n_syn_epi = len(topk_snps & syn_epi_snps) # topk synthetic epistatic SNPs
        n_syn_epi_groups = sum(
            1 for a, b in epi_groups if (a in topk_snps and b in topk_snps)
        ) # topk synthetic epistatic SNP PAIRS
        n_syn_nonlin = len(topk_snps & syn_nonlinear_snps) # topk synthetic nonlinear        
        # TPR = TP / (TP + FN) = TP / n_causal_in_category
        # TP: causal snps found in top-k
        # FP: non-causal snps found in top-k
        # FN: causal snps NOT found in top-k
        # TN: non-causal snps NOT found in top-k
        real_tpr = n_real / len(real_snps) if len(real_snps) > 0 else 0
        gwas_tpr = n_gwas / len(gwas_sig_snps) if len(gwas_sig_snps) > 0 else 0
        syn_tpr = n_syn / len(syn_snps) if len(syn_snps) > 0 else 0
        syn_add_tpr = n_syn_add / len(syn_add_snps) if len(syn_add_snps) > 0 else 0
        syn_nonlin_tpr = n_syn_nonlin / len(syn_nonlinear_snps) if len(syn_nonlinear_snps) > 0 else 0
        syn_dom_tpr = n_syn_dom / len(syn_dom_snps) if len(syn_dom_snps) > 0 else 0
        syn_rec_tpr = n_syn_rec / len(syn_rec_snps) if len(syn_rec_snps) > 0 else 0
        syn_epi_tpr = n_syn_epi / len(syn_epi_snps) if len(syn_epi_snps) > 0 else 0
        syn_epi_groups_tpr = (n_syn_epi_groups / n_total_epi_groups) if n_total_epi_groups > 0 else 0
        # store results
        results.append({
            'algo': algo,
            # threshold vars
            'quantile': q,
            'percentile': percentile,
            'k': k,
            # topk counts
            'n_real': n_real,
            'n_gwas': n_gwas,
            'n_syn': n_syn,
            'n_syn_add': n_syn_add,
            'n_syn_nonlin': n_syn_nonlin,
            'n_syn_dom': n_syn_dom,
            'n_syn_rec': n_syn_rec,
            'n_syn_epi': n_syn_epi,
            'n_epi_groups': n_syn_epi_groups,
            # recall scores
            'real_tpr': real_tpr,
            'gwas_tpr': gwas_tpr,
            'syn_tpr': syn_tpr,
            'syn_add_tpr': syn_add_tpr,
            'syn_nonlin_tpr': syn_nonlin_tpr,
            'syn_dom_tpr': syn_dom_tpr,
            'syn_rec_tpr': syn_rec_tpr,
            'syn_epi_tpr': syn_epi_tpr,
            'syn_epi_groups_tpr': syn_epi_groups_tpr,
        })
results_df = pd.DataFrame(results)
print(results_df.to_string())

# save results
tpr_results_file = os.path.join(model_dir, 'tpr_results.csv')
results_df.to_csv(tpr_results_file, index=False)
print(f"Recall results saved to: {tpr_results_file}")

In [ ]:
index_cols = ['k', 'percentile', 'quantile']
tpr_syn_add_df = results_df.pivot(index=index_cols, columns='algo', values='syn_add_tpr')
tpr_syn_nonlin_df = results_df.pivot(index=index_cols, columns='algo', values='syn_nonlin_tpr')
tpr_syn_dom_df = results_df.pivot(index=index_cols, columns='algo', values='syn_dom_tpr')
tpr_syn_rec_df = results_df.pivot(index=index_cols, columns='algo', values='syn_rec_tpr')
tpr_syn_epi_df = results_df.pivot(index=index_cols, columns='algo', values='syn_epi_tpr')
tpr_syn_epi_groups_df = results_df.pivot(index=index_cols, columns='algo', values='syn_epi_groups_tpr')

In [ ]:
# smaller plot
fig, axs = plt.subplots(1, 3, figsize=(8.0, 3.0), sharey=True, dpi=300)
# fig, axs = plt.subplots(1, 3, figsize=(12.4, 4.2), sharey=True, dpi=300)

non_sg_algos = [algo for algo in tpr_syn_dom_df.columns if not algo.endswith('SG') and algo.lower() != 'gwas']
sg_algos = [algo for algo in tpr_syn_dom_df.columns if algo.endswith('SG')]

# DOM
mean_non_sg = tpr_syn_dom_df[non_sg_algos].mean(axis=1)
std_non_sg = tpr_syn_dom_df[non_sg_algos].std(axis=1)
mean_sg = tpr_syn_dom_df[sg_algos].mean(axis=1)
std_sg = tpr_syn_dom_df[sg_algos].std(axis=1)

axs[0].plot(mean_non_sg.index.get_level_values('quantile'), 
            mean_non_sg, label='without SmoothGrad', color='blue')
axs[0].fill_between(mean_non_sg.index.get_level_values('quantile'), 
                    mean_non_sg - std_non_sg, 
                    mean_non_sg + std_non_sg, 
                    color='blue', alpha=0.2)
axs[0].plot(mean_sg.index.get_level_values('quantile'), 
            mean_sg, label='with SmoothGrad', color='orange')
axs[0].fill_between(mean_sg.index.get_level_values('quantile'), 
                    mean_sg - std_sg, 
                    mean_sg + std_sg, 
                    color='orange', alpha=0.2)
axs[0].plot(tpr_syn_dom_df['gwas'].index.get_level_values('quantile'), 
            tpr_syn_dom_df['gwas'], label='GWAS', color='green')
axs[0].set_xlabel('quantile')
axs[0].set_ylabel('recall')
axs[0].set_title('(A) Dominant')
axs[0].legend(fontsize='small', loc='lower right')
axs[0].set_ylim(0, 1.01)
axs[0].set_xlim(tpr_syn_dom_df.index.get_level_values('quantile').min(), 
                tpr_syn_dom_df.index.get_level_values('quantile').max())
axs[0].grid(True, alpha=0.5)

# REC
mean_non_sg = tpr_syn_rec_df[non_sg_algos].mean(axis=1)
std_non_sg = tpr_syn_rec_df[non_sg_algos].std(axis=1)
mean_sg = tpr_syn_rec_df[sg_algos].mean(axis=1)
std_sg = tpr_syn_rec_df[sg_algos].std(axis=1)
axs[1].plot(mean_non_sg.index.get_level_values('quantile'), 
            mean_non_sg, label='without SmoothGrad', color='blue')
axs[1].fill_between(mean_non_sg.index.get_level_values('quantile'), 
                    mean_non_sg - std_non_sg, 
                    mean_non_sg + std_non_sg, 
                    color='blue', alpha=0.2)
axs[1].plot(mean_sg.index.get_level_values('quantile'), 
            mean_sg, label='with SmoothGrad', color='orange')
axs[1].fill_between(mean_sg.index.get_level_values('quantile'), 
                    mean_sg - std_sg, 
                    mean_sg + std_sg, 
                    color='orange', alpha=0.2)
axs[1].plot(tpr_syn_rec_df['gwas'].index.get_level_values('quantile'), 
            tpr_syn_rec_df['gwas'], label='GWAS', color='green')
axs[1].set_xlabel('quantile')
axs[1].set_title('(B) Recessive')
axs[1].legend(fontsize='small')
axs[1].set_ylim(0, 1.01)
axs[1].set_xlim(tpr_syn_rec_df.index.get_level_values('quantile').min(), 
                tpr_syn_rec_df.index.get_level_values('quantile').max())
axs[1].grid(True, alpha=0.5)

# EPI
mean_non_sg = tpr_syn_epi_df[non_sg_algos].mean(axis=1)
std_non_sg = tpr_syn_epi_df[non_sg_algos].std(axis=1)
mean_sg = tpr_syn_epi_df[sg_algos].mean(axis=1)
std_sg = tpr_syn_epi_df[sg_algos].std(axis=1)
axs[2].plot(mean_non_sg.index.get_level_values('quantile'), 
            mean_non_sg, label='without SmoothGrad', color='blue')
axs[2].fill_between(mean_non_sg.index.get_level_values('quantile'), 
                    mean_non_sg - std_non_sg, 
                    mean_non_sg + std_non_sg, 
                    color='blue', alpha=0.2)
axs[2].plot(mean_sg.index.get_level_values('quantile'), 
            mean_sg, label='with SmoothGrad', color='orange')
axs[2].fill_between(mean_sg.index.get_level_values('quantile'), 
                    mean_sg - std_sg, 
                    mean_sg + std_sg, 
                    color='orange', alpha=0.2)
axs[2].plot(tpr_syn_epi_df['gwas'].index.get_level_values('quantile'), 
            tpr_syn_epi_df['gwas'], label='GWAS', color='green')
axs[2].set_xlabel('quantile')
axs[2].set_title('(C) Epistatic')
axs[2].legend(fontsize='small')
axs[2].grid(True, alpha=0.5)
axs[2].set_ylim(0, 1.01)
axs[2].set_xlim(tpr_syn_epi_df.index.get_level_values('quantile').min(), 
                tpr_syn_epi_df.index.get_level_values('quantile').max())

quantiles_clean = np.round(sorted(tpr_syn_dom_df.index.get_level_values('quantile').values), 2)
# keep even quantiles for x-ticks
even_quantiles_clean = [float(q) for q in quantiles_clean if int(q*100) % 5 == 0]
for ax in axs:
    ax.set_xticks(even_quantiles_clean)
    ax.set_xticklabels([f"{q:.2f}" for q in even_quantiles_clean],)

plot_save_path = f"{os.path.splitext(attrs_path)[0]}_recall_plot.tiff"
plt.savefig(
    plot_save_path,
    dpi=300,
    bbox_inches='tight',
    format='tiff',
    pil_kwargs={"compression": "tiff_lzw"}  # LZW compression to reduce file size
)
print(f"Plot saved to: {plot_save_path}")
plt.tight_layout()
plt.show()